
###coalesce() in Databricks.

1. In Databricks, coalesce() is used to reduce the number of partitions in a DataFrame without performing a full shuffle in typical cases.

2. It is especially useful when you want to reduce the number of output files while writing data.

###Create Dataframe For Csv file


In [0]:
Hotel_Coalesce_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotels.csv")
display(Hotel_Coalesce_df)

###Cleaning Column for dataframe

In [0]:
import re

for old_col in Hotel_Coalesce_df.columns:
    new_col = re.sub(r"[^a-zA-Z0-9_]", "_", old_col)
    new_col = new_col.strip('_').lower()
    Hotel_Coalesce_df = Hotel_Coalesce_df.withColumnRenamed(old_col, new_col)
display(Hotel_Coalesce_df)

####Write Data in Delta fomrat on Volume.

In [0]:
Hotel_Coalesce_df.coalesce(5).write.format("delta").mode("overwrite").save("/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotel_coalesce")

###Display Data file

In [0]:
path = dbutils.fs.ls("/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotel_coalesce")
print(path)
display(dbutils.fs.ls("/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotel_coalesce"))

### Difference between coalesce() and repartition()

| Feature | coalesce() | repartition() |
| --- | --- | --- |
| Shuffle | Usually avoids full shuffle | Performs shuffle |
| Increase partitions | No | Yes |
| Decrease partitions | Yes | Yes |
| Performance | Usually faster for reducing partitions | More expensive due to shuffle |